# exp438 fixed-U-lattice joint exact HMM — Stage 0

This CPU-only Stage 0 candidate keeps the exp209 joint position/U-rate exact
HMM unchanged except for the coordinate that owns the fixed 0.35-ft position
lattice. The parent keeps a fixed TVT lattice and uses
`delta_TVT = r_U * delta_MD - delta_Z`; this candidate anchors one fixed
absolute-U lattice at the last known Z, uses `delta_U = r_U * delta_MD`, and
evaluates every row at `TVT = U - Z`. In continuous coordinates the two
formulations are identical. The scientific ablation is only the fixed
discrete-lattice phase.

The notebook is fail-closed: implementation approval does not authorize a
Kaggle run. Stage 1, inference, submission, parent-HMM regeneration, and any
grid/noise/emission rescue are disabled.

## Contents

1. Imports and immutable execution contract
2. Notebook-safe paths, SHA, and leakage ledger
3. Fixed32 manifest, saved parent, and target-free raw inputs
4. Fixed-U coordinate and exp209 input preparation
5. Joint fixed-lattice exact forward-backward HMM
6. Numerical contracts and target-free prediction freeze
7. Truth-late persistent-episode and safety readout
8. Stage 0 gates, generated artifacts, and metrics
9. Configuration preview and guarded execution

## 1. Imports and immutable execution contract

In [ ]:
from __future__ import annotations

import gzip
import hashlib
import io
import json
import math
import os
import platform
import resource
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd
import yaml
from numba import njit, set_num_threads

EXPERIMENT_NAME = "exp438_u_state_fixed_lattice_exact_hmm"
PARENT_EXPERIMENT = "exp209_exp072_exp205_joint_exact_parity_fast_cache_generation"
EVIDENCE_EXPERIMENT = "exp408_hmm_message_rate_basin_audit"
SCIENTIFIC_VARIANT = "fixed_u_lattice_joint_rate"
PACKAGE_DIR = Path.cwd()
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORKING_ROOT = Path("/kaggle/working")

FORBIDDEN_DECODER_COLUMNS = frozenset(
    {
        "TVT",
        "tvt_true",
        "error",
        "abs_error",
        "episode_id",
        "start_row_idx",
        "end_row_idx_exclusive",
        "fold",
        "role",
        "hidden_like_role",
    }
)


def get_nested(mapping: Mapping[str, Any], dotted_key: str, default: Any = None) -> Any:
    value: Any = mapping
    for part in dotted_key.split("."):
        if not isinstance(value, Mapping) or part not in value:
            return default
        value = value[part]
    return value


def validate_execution_contract(
    config: Mapping[str, Any],
    *,
    require_run_authorization: bool,
) -> dict[str, int]:
    if get_nested(config, "experiment.name") != EXPERIMENT_NAME:
        raise ValueError("wrong exp438 config")
    if get_nested(config, "experiment.route") != "pf_beam":
        raise ValueError("exp438 route must remain pf_beam")
    if get_nested(config, "lineage.parent") != PARENT_EXPERIMENT:
        raise ValueError("exp438 scientific parent changed")
    if not bool(get_nested(config, "runtime.implementation_approved", False)):
        raise RuntimeError("exp438 implementation is not approved")
    if bool(get_nested(config, "runtime.stage1_approved", True)):
        raise ValueError("Stage 1 must remain disabled during Stage 0")
    if bool(get_nested(config, "runtime.inference_enabled", True)):
        raise ValueError("inference must remain disabled")
    if bool(get_nested(config, "runtime.submission_enabled", True)):
        raise ValueError("submission must remain disabled")
    if str(get_nested(config, "runtime.accelerator")) != "cpu":
        raise ValueError("exp438 is CPU-only")
    if bool(get_nested(config, "runtime.internet", True)):
        raise ValueError("exp438 internet must remain disabled")

    expected = {
        "scientific_variants": 1,
        "reporting_folds": 5,
        "stage0_hmm_well_runs": 32,
        "stage1_max_hmm_well_runs": 773,
        "parent_control_hmm_reruns": 0,
        "fitted_ml_models": 0,
        "lightgbm_configs": 0,
        "trained_ml_folds": 0,
        "boosters": 0,
        "pf_runs": 0,
        "beam_runs": 0,
        "gpu_runs": 0,
    }
    observed = {
        key: int(get_nested(config, f"execution.{key}", -1)) for key in expected
    }
    if observed != expected:
        raise ValueError(f"Stage 0 execution contract changed: {observed} != {expected}")
    if get_nested(config, "execution.selected_stage") != "stage_0_fixed32":
        raise ValueError("selected_stage must remain stage_0_fixed32")
    if bool(get_nested(config, "validation.parent_rerun", True)):
        raise ValueError("saved exp209 prediction must remain the control")
    if require_run_authorization:
        if not bool(get_nested(config, "runtime.run_approved", False)):
            raise RuntimeError(
                "implementation approval does not authorize Kaggle execution"
            )
        if not bool(get_nested(config, "execution.run_hmm", False)):
            raise RuntimeError("execution.run_hmm is false")
    return observed


def validate_scientific_contract(config: Mapping[str, Any]) -> dict[str, Any]:
    fixed = get_nested(config, "model.fixed_from_exp209")
    expected_fixed = {
        "position_grid_step_ft": 0.35,
        "n_rates": 41,
        "rate_span": 0.10,
        "sig_r": 0.002,
        "sig_p": 0.02,
        "effective_position_sigma_ft": 0.1225,
        "position_kernel_cells": 5,
        "momentum": 0.998,
        "emission": "gaussian_typewell_gr",
        "emission_lambda": 1.0,
        "start_sigma_ft": 0.75,
        "initial_rate_sigma": 0.01,
        "band_pad_ft": 100.0,
        "rate_center": "zero",
        "rate_position_integration": "arrival_rate",
        "output": "smoothed_posterior_mean_and_std",
    }
    if fixed != expected_fixed:
        raise ValueError(f"exp209 HMM contract changed: {fixed} != {expected_fixed}")
    candidate = get_nested(config, "model.candidate_position")
    expected_candidate = {
        "lattice": "fixed_absolute_u",
        "grid_formula": "parent_tvt_grid+last_known_Z",
        "grid_reanchoring_after_start": "forbidden",
        "row_adaptive_regridding": "forbidden",
        "transition_mean_formula": "r_current*delta_MD",
        "emission_state_formula": "U_state-Z_current",
        "readout_formula": "E_U-Z_current",
    }
    if candidate != expected_candidate:
        raise ValueError(
            f"fixed-U coordinate contract changed: {candidate} != {expected_candidate}"
        )
    if get_nested(config, "model.candidate_state") != ["u_position", "u_rate"]:
        raise ValueError("candidate state must remain joint (U, U-rate)")
    if get_nested(config, "model.coordinate.continuous_parent_equivalence") != "exact":
        raise ValueError("continuous U/TVT equivalence must remain exact")
    if (
        get_nested(config, "model.coordinate.scientific_difference")
        != "fixed_discrete_lattice_coordinate_only"
    ):
        raise ValueError("exp438 must remain a one-factor lattice-coordinate ablation")
    return {
        "fixed_from_exp209": fixed,
        "candidate_position": candidate,
        "candidate_state": get_nested(config, "model.candidate_state"),
        "forbidden": get_nested(config, "model.forbidden"),
    }

## 2. Notebook-safe paths, SHA, and leakage ledger

In [ ]:
def find_project_root(start: Path = PACKAGE_DIR) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "project.yml").is_file():
            return candidate
    return start


def config_path() -> Path:
    root = find_project_root()
    candidates = (
        root / "experiments" / EXPERIMENT_NAME / "config.yaml",
        PACKAGE_DIR / "config.yaml",
    )
    for candidate in candidates:
        if candidate.is_file():
            value = yaml.safe_load(candidate.read_text()) or {}
            if get_nested(value, "experiment.name") == EXPERIMENT_NAME:
                return candidate
    raise FileNotFoundError("exp438 config.yaml was not found")


def load_config(path: Path | None = None) -> dict[str, Any]:
    resolved = config_path() if path is None else path
    value = yaml.safe_load(resolved.read_text()) or {}
    if not isinstance(value, dict):
        raise ValueError(f"{resolved} must contain a YAML mapping")
    return value


def artifacts_dir() -> Path:
    if KAGGLE_WORKING_ROOT.is_dir():
        target = KAGGLE_WORKING_ROOT / "artifacts"
    else:
        target = find_project_root() / "experiments" / EXPERIMENT_NAME / "artifacts"
    target.mkdir(parents=True, exist_ok=True)
    return target


def metrics_path() -> Path:
    if KAGGLE_WORKING_ROOT.is_dir():
        return KAGGLE_WORKING_ROOT / "metrics.json"
    return find_project_root() / "experiments" / EXPERIMENT_NAME / "metrics.json"


def to_jsonable(value: Any) -> Any:
    if isinstance(value, Mapping):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(item) for item in value]
    if isinstance(value, np.ndarray):
        return [to_jsonable(item) for item in value.tolist()]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        number = float(value)
        return number if math.isfinite(number) else None
    if isinstance(value, Path):
        return str(value)
    try:
        if pd.isna(value) and not isinstance(value, str):
            return None
    except (TypeError, ValueError):
        pass
    return value


def stable_json_bytes(value: Any) -> bytes:
    return json.dumps(
        to_jsonable(value),
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=True,
    ).encode()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_decompressed_csv(path: Path) -> str:
    digest = hashlib.sha256()
    with gzip.open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def array_bundle_sha256(**arrays: np.ndarray) -> str:
    digest = hashlib.sha256()
    for name in sorted(arrays):
        array = np.ascontiguousarray(arrays[name])
        digest.update(name.encode())
        digest.update(str(array.dtype).encode())
        digest.update(np.asarray(array.shape, dtype=np.int64).tobytes())
        digest.update(array.tobytes(order="C"))
    return digest.hexdigest()


def logical_frame_sha256(frame: pd.DataFrame) -> str:
    normalized = frame.copy()
    for column in normalized.columns:
        if pd.api.types.is_float_dtype(normalized[column]):
            normalized[column] = normalized[column].astype(np.float64)
        elif pd.api.types.is_integer_dtype(normalized[column]):
            normalized[column] = normalized[column].astype(np.int64)
        else:
            normalized[column] = normalized[column].astype(str)
    payload = normalized.to_csv(index=False, lineterminator="\n").encode()
    return hashlib.sha256(payload).hexdigest()


def write_json(path: Path, payload: Any) -> dict[str, Any]:
    path.write_text(json.dumps(to_jsonable(payload), indent=2, sort_keys=True) + "\n")
    return {
        "path": str(path),
        "sha256": sha256_file(path),
        "bytes": path.stat().st_size,
    }


def write_csv(path: Path, frame: pd.DataFrame) -> dict[str, Any]:
    frame.to_csv(path, index=False, lineterminator="\n")
    return {
        "path": str(path),
        "sha256": sha256_file(path),
        "logical_sha256": logical_frame_sha256(frame),
        "rows": len(frame),
    }


def write_deterministic_gzip_csv(path: Path, frame: pd.DataFrame) -> dict[str, Any]:
    raw = path.open("wb")
    compressed = gzip.GzipFile(
        filename="",
        mode="wb",
        fileobj=raw,
        compresslevel=1,
        mtime=0,
    )
    text = io.TextIOWrapper(compressed, encoding="utf-8", newline="")
    try:
        frame.to_csv(text, index=False, lineterminator="\n")
    finally:
        text.flush()
        text.close()
        compressed.close()
        raw.close()
    readback = pd.read_csv(path, float_precision="round_trip")
    return {
        "path": str(path),
        "raw_sha256": sha256_file(path),
        "decompressed_sha256": sha256_decompressed_csv(path),
        "logical_sha256": logical_frame_sha256(frame),
        "readback_logical_sha256": logical_frame_sha256(readback),
        "rows": len(frame),
    }


def peak_rss_gb() -> float:
    value = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
    if platform.system() == "Darwin":
        return value / (1024**3)
    return value / (1024**2)


def runtime_versions() -> dict[str, Any]:
    import numba

    return {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "numba": numba.__version__,
        "machine": platform.machine(),
        "processor": platform.processor(),
        "cpu_count": os.cpu_count(),
    }


def resolve_bootstrap_asset(filename: str, local_path: str) -> Path:
    candidates = (
        PACKAGE_DIR / filename,
        PACKAGE_DIR / "assets" / filename,
        find_project_root() / local_path,
    )
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"bootstrap asset not found: {filename}")


def resolve_unique_file(
    *,
    filename: str,
    candidates: Sequence[str],
    patterns: Sequence[str],
) -> Path:
    root = find_project_root()
    matches: list[Path] = []
    for item in candidates:
        candidate = Path(item)
        if not candidate.is_absolute():
            candidate = root / candidate
        if candidate.is_file():
            matches.append(candidate)
        elif (candidate / filename).is_file():
            matches.append(candidate / filename)
    if KAGGLE_INPUT_ROOT.is_dir():
        for pattern in patterns:
            matches.extend(KAGGLE_INPUT_ROOT.glob(pattern))
    unique = sorted({path.resolve() for path in matches if path.is_file()})
    if not unique:
        raise FileNotFoundError(f"could not resolve {filename}")
    if len(unique) > 1:
        hashes = {sha256_file(path) for path in unique}
        if len(hashes) != 1:
            raise RuntimeError(f"multiple non-identical files found for {filename}")
    return unique[0]


@dataclass
class LeakageLedger:
    frozen_wells: set[str] = field(default_factory=set)
    scope_rows: int = 0
    target_free_rows: int = 0
    truth_rows_before_all_freeze: int = 0
    role_fold_rows_before_all_freeze: int = 0
    episode_rows_before_all_freeze: int = 0
    truth_rows_after_all_freeze: int = 0
    role_fold_rows_after_all_freeze: int = 0
    episode_rows_after_all_freeze: int = 0
    expected_wells: int = 32

    @property
    def all_frozen(self) -> bool:
        return len(self.frozen_wells) == self.expected_wells

    def record_scope(self, rows: int) -> None:
        self.scope_rows += int(rows)

    def record_target_free(self, rows: int) -> None:
        self.target_free_rows += int(rows)

    def freeze(self, well: str) -> None:
        self.frozen_wells.add(str(well))

    def record_truth_late(self, rows: int) -> None:
        if not self.all_frozen:
            self.truth_rows_before_all_freeze += int(rows)
            raise RuntimeError("truth was read before all fixed32 predictions were frozen")
        self.truth_rows_after_all_freeze += int(rows)

    def record_role_fold_late(self, rows: int) -> None:
        if not self.all_frozen:
            self.role_fold_rows_before_all_freeze += int(rows)
            raise RuntimeError("role/fold was read before all fixed32 predictions were frozen")
        self.role_fold_rows_after_all_freeze += int(rows)

    def record_episode_late(self, rows: int) -> None:
        if not self.all_frozen:
            self.episode_rows_before_all_freeze += int(rows)
            raise RuntimeError("episodes were read before all fixed32 predictions were frozen")
        self.episode_rows_after_all_freeze += int(rows)

## 3. Fixed32 manifest, saved parent, and target-free raw inputs

Before all 32 candidate predictions and diagnostic SHAs are frozen, the
manifest reader opens only `well`, `prefix_rows`, and `suffix_rows`. Role and
fold are reread through a separately guarded function after freeze.

In [ ]:
def train_data_dir(config: Mapping[str, Any]) -> Path:
    if KAGGLE_INPUT_ROOT.is_dir():
        fixed = (
            KAGGLE_INPUT_ROOT / "rogii-wellbore-geology-prediction" / "train",
            KAGGLE_INPUT_ROOT
            / "competitions"
            / "rogii-wellbore-geology-prediction"
            / "train",
        )
        for candidate in fixed:
            if next(candidate.glob("*__horizontal_well.csv"), None) is not None:
                return candidate
        first = next(KAGGLE_INPUT_ROOT.glob("**/*__horizontal_well.csv"), None)
        if first is not None:
            return first.parent
    return find_project_root() / str(get_nested(config, "data.train_dir"))


def fixed32_manifest_path(config: Mapping[str, Any]) -> tuple[Path, str]:
    spec = get_nested(config, "data.fixed32_manifest")
    path = resolve_bootstrap_asset(str(spec["filename"]), str(spec["local"]))
    observed = sha256_file(path)
    if observed != str(spec["expected_sha256"]):
        raise ValueError(f"fixed32 manifest SHA changed: {observed}")
    return path, observed


def load_fixed32_target_free_scope(
    config: Mapping[str, Any],
    ledger: LeakageLedger,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    path, observed = fixed32_manifest_path(config)
    frame = pd.read_csv(
        path,
        usecols=["well", "prefix_rows", "suffix_rows"],
        dtype={"well": str},
    )
    if len(frame) != 32 or frame["well"].nunique() != 32:
        raise ValueError("fixed32 manifest must contain 32 unique wells")
    ledger.record_scope(len(frame))
    frame = frame.sort_values("well", kind="mergesort").reset_index(drop=True)
    return frame, {
        "path": str(path),
        "sha256": observed,
        "rows": len(frame),
        "target_free_logical_sha256": logical_frame_sha256(frame),
    }


def load_fixed32_identity_after_all_freeze(
    config: Mapping[str, Any],
    ledger: LeakageLedger,
) -> pd.DataFrame:
    path, _ = fixed32_manifest_path(config)
    frame = pd.read_csv(path, dtype={"well": str, "matched_persistent_well": str})
    ledger.record_role_fold_late(len(frame))
    if len(frame) != 32 or frame["well"].nunique() != 32:
        raise ValueError("fixed32 identity changed")
    if frame["role"].value_counts().to_dict() != {"persistent": 16, "control": 16}:
        raise ValueError("fixed32 role counts changed")
    if set(frame["fold"].astype(int)) != {0, 1, 2, 3, 4}:
        raise ValueError("fixed32 fold coverage changed")
    return frame.sort_values("well", kind="mergesort").reset_index(drop=True)


def parent_row_indices_from_cache_ids(frame: pd.DataFrame) -> np.ndarray:
    row_indices = np.empty(len(frame), dtype=np.int64)
    for offset, (well, identifier) in enumerate(
        zip(frame["well"].astype(str), frame["id"].astype(str), strict=True)
    ):
        prefix = f"{well}_"
        if not identifier.startswith(prefix):
            raise ValueError(
                f"saved parent id does not start with exact well prefix: {identifier}"
            )
        suffix = identifier[len(prefix) :]
        if not suffix.isdigit():
            raise ValueError(f"saved parent id has invalid row suffix: {identifier}")
        row_indices[offset] = int(suffix)
    return row_indices


def parent_cache_ids_for_rows(well: str, row_indices: np.ndarray) -> np.ndarray:
    well = str(well)
    rows = np.asarray(row_indices, dtype=np.int64)
    if not well or rows.ndim != 1 or np.any(rows < 0):
        raise ValueError("invalid well or row indices for parent cache ids")
    return np.asarray([f"{well}_{int(row)}" for row in rows], dtype=str)


def load_saved_parent_predictions(
    config: Mapping[str, Any],
    target_wells: set[str],
    expected_rows: int,
    ledger: LeakageLedger,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    spec = get_nested(config, "data.exp209_saved_control")
    path = resolve_unique_file(
        filename=str(spec["filename"]),
        candidates=[str(value) for value in spec["candidates"]],
        patterns=[str(value) for value in spec["patterns"]],
    )
    decompressed = sha256_decompressed_csv(path)
    if decompressed != str(spec["expected_decompressed_sha256"]):
        raise ValueError(f"saved exp209 decompressed SHA changed: {decompressed}")
    columns = ["id", "well", str(spec["prediction_column"])]
    pieces: list[pd.DataFrame] = []
    for chunk in pd.read_csv(
        path,
        usecols=columns,
        dtype={"id": str, "well": str},
        chunksize=200_000,
    ):
        selected = chunk.loc[chunk["well"].isin(target_wells)]
        if not selected.empty:
            pieces.append(selected)
    if not pieces:
        raise ValueError("saved exp209 control has no fixed32 rows")
    frame = pd.concat(pieces, ignore_index=True)
    frame = frame.rename(columns={str(spec["prediction_column"]): "parent_prediction"})
    frame["row_idx"] = parent_row_indices_from_cache_ids(frame)
    frame["parent_prediction"] = pd.to_numeric(
        frame["parent_prediction"], errors="raise"
    )
    frame = frame.sort_values(["well", "row_idx"], kind="mergesort").reset_index(drop=True)
    if len(frame) != expected_rows:
        raise ValueError(f"saved parent rows={len(frame)}/{expected_rows}")
    if frame.duplicated(["well", "row_idx"]).any():
        raise ValueError("saved parent keys are not unique")
    ledger.record_target_free(len(frame))
    return frame, {
        "path": str(path),
        "raw_sha256": sha256_file(path),
        "decompressed_sha256": decompressed,
        "rows": len(frame),
    }


def load_target_free_well(
    well: str,
    raw_dir: Path,
    ledger: LeakageLedger,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    horizontal_path = raw_dir / f"{well}__horizontal_well.csv"
    typewell_path = raw_dir / f"{well}__typewell.csv"
    horizontal = pd.read_csv(
        horizontal_path,
        usecols=lambda column: str(column) != "TVT",
    )
    forbidden = FORBIDDEN_DECODER_COLUMNS.intersection(horizontal.columns)
    if forbidden:
        raise ValueError(f"{well}: decoder input contains {sorted(forbidden)}")
    typewell = pd.read_csv(typewell_path).sort_values("TVT").reset_index(drop=True)
    ledger.record_target_free(len(horizontal) + len(typewell))
    return horizontal, typewell

## 4. Fixed-U coordinate and exp209 input preparation

The parent TVT grid is built exactly once from the exp209 bounds. The
candidate grid is `parent_tvt_grid + last_known_Z` and is never reanchored.
Each row receives a view of that fixed U grid in TVT coordinates only for the
Type Well GR emission.

In [ ]:
def robust_initial_rate(
    known_prefix: pd.DataFrame,
    window_rows: int = 30,
    *,
    min_valid_steps: int = 3,
    fallback_rate: float = 0.0,
) -> tuple[float, int, int]:
    tail = known_prefix.tail(int(window_rows))
    tvt = pd.to_numeric(tail["TVT_input"], errors="coerce").to_numpy(np.float64)
    z = pd.to_numeric(tail["Z"], errors="coerce").to_numpy(np.float64)
    md = pd.to_numeric(tail["MD"], errors="coerce").to_numpy(np.float64)
    dtvt = np.diff(tvt)
    dz = np.diff(z)
    dmd = np.diff(md)
    valid = np.isfinite(dtvt) & np.isfinite(dz) & np.isfinite(dmd) & (dmd > 0.0)
    valid_steps = int(valid.sum())
    if valid_steps < int(min_valid_steps):
        return float(fallback_rate), int(len(tail)), valid_steps
    rate = float(np.median((dtvt[valid] + dz[valid]) / dmd[valid]))
    if not np.isfinite(rate):
        rate = float(fallback_rate)
    return rate, int(len(tail)), valid_steps


def prefix_stats(
    horizontal: pd.DataFrame,
    typewell_tvt: np.ndarray,
    typewell_gr: np.ndarray,
    tail_n: int = 30,
) -> tuple[float, float, float, float, int, int]:
    known = horizontal.loc[horizontal["TVT_input"].notna()]
    known_gr = known["GR"].to_numpy(np.float64)
    known_tvt = known["TVT_input"].to_numpy(np.float64)
    typewell_at_known = np.interp(known_tvt, typewell_tvt, typewell_gr)
    valid = np.isfinite(known_gr) & np.isfinite(typewell_at_known)
    if valid.sum() >= 20 and np.std(typewell_at_known[valid]) > 1.0e-6:
        cal_a, cal_b = np.polyfit(typewell_at_known[valid], known_gr[valid], 1)
    elif valid.any():
        cal_a = 1.0
        cal_b = float(np.nanmean(known_gr) - np.nanmean(typewell_at_known))
    else:
        cal_a, cal_b = 1.0, 0.0
    residual = known_gr[valid] - (cal_a * typewell_at_known[valid] + cal_b)
    if valid.sum() > 20:
        sigma = float(
            np.clip(
                1.4826 * np.median(np.abs(residual - np.median(residual))),
                8.0,
                60.0,
            )
        )
    else:
        sigma = 30.0
    init_rate, effective_rows, valid_steps = robust_initial_rate(known, tail_n)
    return (
        float(cal_a),
        float(cal_b),
        sigma,
        init_rate,
        effective_rows,
        valid_steps,
    )


def prepare_hmm_inputs(
    horizontal: pd.DataFrame,
    typewell: pd.DataFrame,
    hmm: Mapping[str, Any],
) -> dict[str, Any]:
    required_horizontal = {"MD", "Z", "GR", "TVT_input"}
    required_typewell = {"TVT", "GR"}
    if not required_horizontal.issubset(horizontal.columns):
        raise ValueError("horizontal input schema changed")
    if not required_typewell.issubset(typewell.columns):
        raise ValueError("typewell input schema changed")
    if "TVT" in horizontal.columns:
        raise ValueError("unknown-suffix TVT reached HMM preparation")

    typewell_tvt = typewell["TVT"].to_numpy(np.float64)
    typewell_gr = typewell["GR"].ffill().bfill().to_numpy(np.float64)
    known = horizontal.loc[horizontal["TVT_input"].notna()]
    eval_rows = horizontal.loc[horizontal["TVT_input"].isna()]
    if len(known) < 4 or len(eval_rows) == 0:
        raise ValueError("expected a visible prefix and non-empty suffix")
    cal_a, cal_b, robust_sigma, init_rate, rate_rows, valid_steps = prefix_stats(
        horizontal, typewell_tvt, typewell_gr
    )
    known_tvt = known["TVT_input"].to_numpy(np.float64)
    typewell_at_known = np.interp(known_tvt, typewell_tvt, typewell_gr)
    residual = known["GR"].fillna(0).to_numpy(np.float64) - typewell_at_known
    gr_sigma = float(np.clip(np.nanstd(residual), 10.0, 60.0))

    step = float(hmm["position_grid_step_ft"])
    last = known.iloc[-1]
    last_tvt = float(last["TVT_input"])
    last_z = float(last["Z"])
    grid_min = max(
        float(typewell_tvt.min()) - 40.0,
        last_tvt - float(hmm["band_pad_ft"]),
    )
    grid_max = min(
        float(typewell_tvt.max()) + 40.0,
        last_tvt + float(hmm["band_pad_ft"]),
    )
    parent_tvt_grid = np.arange(
        grid_min,
        grid_max + step,
        step,
        dtype=np.float64,
    )
    u_grid = parent_tvt_grid + last_z
    md = eval_rows["MD"].to_numpy(np.float64)
    z = eval_rows["Z"].to_numpy(np.float64)
    raw_gr = eval_rows["GR"].to_numpy(np.float64)
    gr_fill = float(np.nanmean(typewell_gr))
    gr = (
        horizontal["GR"]
        .interpolate(limit_direction="both")
        .fillna(gr_fill)
        .to_numpy(np.float64)[eval_rows.index]
    )
    dm = np.maximum(np.diff(np.concatenate([[float(last["MD"])], md])), 1.0)
    dz = np.diff(np.concatenate([[last_z], z]))
    row_tvt_grid = u_grid[None, :] - z[:, None]
    row_gr_grid = np.empty_like(row_tvt_grid)
    for row in range(len(row_tvt_grid)):
        row_gr_grid[row] = np.interp(
            row_tvt_grid[row],
            typewell_tvt,
            typewell_gr,
        )
    zscore = (gr[:, None] - row_gr_grid) / gr_sigma
    emission_ll_exact = -0.5 * np.minimum(zscore**2, 600.0)
    emission_ll = emission_ll_exact.astype(np.float32)
    span = max(float(hmm["rate_span"]), abs(init_rate) + 0.04)
    rates = np.linspace(-span, span, int(hmm["n_rates"]), dtype=np.float64)
    return {
        "emission_ll": emission_ll,
        "emission_ll_exact": emission_ll_exact,
        "dm": dm,
        "dz": dz,
        "z": z,
        "gr": gr,
        "typewell_tvt": typewell_tvt,
        "typewell_gr": typewell_gr,
        "parent_tvt_grid": parent_tvt_grid,
        "u_grid": u_grid,
        "row_tvt_grid": row_tvt_grid,
        "rates": rates,
        "start_p": float((last_tvt - grid_min) / step),
        "r0": float(init_rate),
        "eval_index": eval_rows.index.to_numpy(np.int64),
        "raw_gr_missing": ~np.isfinite(raw_gr),
        "last_known_tvt": last_tvt,
        "last_known_md": float(last["MD"]),
        "last_known_z": last_z,
        "prefix_rows": int(len(known)),
        "prefix_sigma": gr_sigma,
        "prefix_ir": init_rate,
        "initial_rate_effective_rows": int(rate_rows),
        "initial_rate_valid_steps": int(valid_steps),
        "cal_a": cal_a,
        "cal_b": cal_b,
        "robust_sigma_unused": robust_sigma,
    }


def coordinate_contract_from_prepared(
    prepared: Mapping[str, Any],
) -> dict[str, Any]:
    u_grid = np.asarray(prepared["u_grid"], dtype=np.float64)
    z = np.asarray(prepared["z"], dtype=np.float64)
    row_tvt_grid = np.asarray(prepared["row_tvt_grid"], dtype=np.float64)
    rates = np.asarray(prepared["rates"], dtype=np.float64)
    dm = np.asarray(prepared["dm"], dtype=np.float64)
    dz = np.asarray(prepared["dz"], dtype=np.float64)
    direct_tvt = u_grid[None, :] - z[:, None]
    tvt_identity = float(np.max(np.abs(row_tvt_grid - direct_tvt)))
    delta_u = dm[:, None] * rates[None, :]
    parent_delta_tvt = delta_u - dz[:, None]
    candidate_delta_tvt = delta_u - dz[:, None]
    transition_identity = float(
        np.max(np.abs(parent_delta_tvt - candidate_delta_tvt))
    )
    typewell_tvt = np.asarray(prepared["typewell_tvt"], dtype=np.float64)
    typewell_gr = np.asarray(prepared["typewell_gr"], dtype=np.float64)
    gr = np.asarray(prepared["gr"], dtype=np.float64)
    sigma = float(prepared["prefix_sigma"])
    direct_emission = np.empty_like(row_tvt_grid, dtype=np.float64)
    for row in range(len(row_tvt_grid)):
        direct_grid = np.interp(
            u_grid - z[row],
            typewell_tvt,
            typewell_gr,
        )
        direct_zscore = (gr[row] - direct_grid) / sigma
        direct_emission[row] = -0.5 * np.minimum(direct_zscore**2, 600.0)
    emission_identity = float(
        np.max(
            np.abs(
                direct_emission
                - np.asarray(prepared["emission_ll_exact"], dtype=np.float64)
            )
        )
    )
    return {
        "tvt_equals_u_minus_z_max_abs_ft": tvt_identity,
        "transition_coordinate_identity_max_abs_ft": transition_identity,
        "emission_coordinate_identity_max_abs": emission_identity,
        "rows": len(z),
        "position_states": len(u_grid),
        "rate_states": len(rates),
        "sha256": array_bundle_sha256(
            eval_index=np.asarray(prepared["eval_index"], dtype=np.int64),
            u_grid=u_grid,
            z=z,
            row_tvt_grid=row_tvt_grid,
            emission_ll=np.asarray(prepared["emission_ll"], dtype=np.float32),
        ),
    }

## 5. Joint fixed-lattice exact forward-backward HMM

`_hmm2_fb_fixed_lattice` is coordinate-generic. `transition_coordinate_delta`
is `delta_Z` for the fixed-TVT parent equation and all zeros for the fixed-U
candidate. The rate process, arrival-rate position mean, five-cell support,
emission, priors, and sum-product forward/backward order otherwise match
exp209.

In [ ]:
@njit(cache=True, nogil=True)
def rate_kernel_probabilities(
    rates: np.ndarray,
    dm: float,
    sig_r: float,
    momentum: float,
) -> np.ndarray:
    rate_count = len(rates)
    rate_step = rates[1] - rates[0]
    sigma_rate_step = sig_r * np.sqrt(dm)
    rate_variance_cells = (sigma_rate_step / rate_step) ** 2
    kernel = np.empty((rate_count, 3), np.float64)
    for rate_index in range(rate_count):
        mean_rate_move = (
            -(1.0 - momentum) * rates[rate_index] * dm / rate_step
        )
        p_plus = max(
            0.5 * (rate_variance_cells + mean_rate_move),
            1.0e-12,
        )
        p_minus = max(
            0.5 * (rate_variance_cells - mean_rate_move),
            1.0e-12,
        )
        total = p_plus + p_minus
        if total > 0.9:
            p_plus *= 0.9 / total
            p_minus *= 0.9 / total
        kernel[rate_index, 0] = p_minus
        kernel[rate_index, 1] = 1.0 - p_plus - p_minus
        kernel[rate_index, 2] = p_plus
    return kernel


@njit(cache=True, nogil=True)
def position_kernel_probabilities(
    mean_shift: float,
    step: float,
    sig_p: float,
) -> tuple[np.ndarray, np.ndarray]:
    sigma_position = max(sig_p, 0.35 * step)
    center = int(np.floor(mean_shift / step + 0.5))
    offsets = np.empty(5, np.int64)
    log_weights = np.empty(5, np.float64)
    for kernel_index in range(5):
        offset = center - 2 + kernel_index
        delta = offset * step - mean_shift
        offsets[kernel_index] = offset
        log_weights[kernel_index] = -0.5 * (delta / sigma_position) ** 2
    maximum = np.max(log_weights)
    weights = np.exp(log_weights - maximum)
    weights /= np.sum(weights)
    return offsets, weights


@njit(cache=True, nogil=True)
def _hmm2_fb_fixed_lattice(
    emission_ll: np.ndarray,
    dm: np.ndarray,
    transition_coordinate_delta: np.ndarray,
    step: float,
    rates: np.ndarray,
    sig_r: float,
    sig_p: float,
    start_p: float,
    start_sig: float,
    r0: float,
    r0_sig: float,
    emission_lambda: float,
    momentum: float,
) -> tuple[np.ndarray, np.ndarray, float, float]:
    time_count, position_count = emission_ll.shape
    rate_count = len(rates)
    negative = np.float32(-1.0e18)
    alpha = np.full(
        (time_count, position_count, rate_count),
        negative,
        np.float32,
    )
    previous = np.full((position_count, rate_count), negative, np.float32)
    for position_index in range(position_count):
        delta_position = (position_index - start_p) * step
        initial_position_logp = -0.5 * (delta_position / start_sig) ** 2
        if initial_position_logp < -60.0:
            continue
        for rate_index in range(rate_count):
            delta_rate = (rates[rate_index] - r0) / r0_sig
            previous[position_index, rate_index] = np.float32(
                initial_position_logp - 0.5 * delta_rate * delta_rate
            )

    rate_updated = np.empty((position_count, rate_count), np.float32)
    current = np.empty((position_count, rate_count), np.float32)
    for time_index in range(time_count):
        rate_kernel = rate_kernel_probabilities(
            rates,
            dm[time_index],
            sig_r,
            momentum,
        )
        rate_log_kernel = np.log(rate_kernel)
        for position_index in range(position_count):
            for destination_rate in range(rate_count):
                best = negative
                first_source = max(destination_rate - 1, 0)
                last_source = min(destination_rate + 1, rate_count - 1)
                for source_rate in range(first_source, last_source + 1):
                    value = (
                        previous[position_index, source_rate]
                        + rate_log_kernel[
                            source_rate,
                            destination_rate - source_rate + 1,
                        ]
                    )
                    if value > best:
                        best = value
                if best > negative / 2:
                    total = 0.0
                    for source_rate in range(first_source, last_source + 1):
                        total += np.exp(
                            previous[position_index, source_rate]
                            + rate_log_kernel[
                                source_rate,
                                destination_rate - source_rate + 1,
                            ]
                            - best
                        )
                    rate_updated[position_index, destination_rate] = np.float32(
                        best + np.log(total)
                    )
                else:
                    rate_updated[position_index, destination_rate] = negative

        for destination_rate in range(rate_count):
            mean_shift = (
                rates[destination_rate] * dm[time_index]
                - transition_coordinate_delta[time_index]
            )
            offsets, position_weights = position_kernel_probabilities(
                mean_shift,
                step,
                sig_p,
            )
            position_log_weights = np.log(position_weights)
            for destination_position in range(position_count):
                best = negative
                for kernel_index in range(5):
                    source_position = (
                        destination_position - offsets[kernel_index]
                    )
                    if 0 <= source_position < position_count:
                        value = (
                            rate_updated[source_position, destination_rate]
                            + position_log_weights[kernel_index]
                        )
                        if value > best:
                            best = value
                if best > negative / 2:
                    total = 0.0
                    for kernel_index in range(5):
                        source_position = (
                            destination_position - offsets[kernel_index]
                        )
                        if 0 <= source_position < position_count:
                            total += np.exp(
                                rate_updated[source_position, destination_rate]
                                + position_log_weights[kernel_index]
                                - best
                            )
                    current[destination_position, destination_rate] = np.float32(
                        best
                        + np.log(total)
                        + emission_lambda
                        * emission_ll[time_index, destination_position]
                    )
                else:
                    current[destination_position, destination_rate] = negative
        for position_index in range(position_count):
            for rate_index in range(rate_count):
                alpha[time_index, position_index, rate_index] = current[
                    position_index, rate_index
                ]
                previous[position_index, rate_index] = current[
                    position_index, rate_index
                ]

    best = negative
    for position_index in range(position_count):
        for rate_index in range(rate_count):
            best = max(best, alpha[-1, position_index, rate_index])
    total = 0.0
    for position_index in range(position_count):
        for rate_index in range(rate_count):
            total += np.exp(alpha[-1, position_index, rate_index] - best)
    log_likelihood = float(best) + np.log(total)

    posterior_position = np.zeros((time_count, position_count), np.float64)
    posterior_rate = np.zeros((time_count, rate_count), np.float64)
    beta_next = np.zeros((position_count, rate_count), np.float32)
    values = alpha[-1] + beta_next
    best = np.max(values)
    total = 0.0
    for position_index in range(position_count):
        for rate_index in range(rate_count):
            total += np.exp(values[position_index, rate_index] - best)
    for position_index in range(position_count):
        for rate_index in range(rate_count):
            probability = (
                np.exp(values[position_index, rate_index] - best) / total
            )
            posterior_position[-1, position_index] += probability
            posterior_rate[-1, rate_index] += probability

    beta_current = np.empty((position_count, rate_count), np.float32)
    beta_position = np.empty((position_count, rate_count), np.float32)
    for time_index in range(time_count - 1, 0, -1):
        rate_kernel = rate_kernel_probabilities(
            rates,
            dm[time_index],
            sig_r,
            momentum,
        )
        rate_log_kernel = np.log(rate_kernel)
        for destination_rate in range(rate_count):
            mean_shift = (
                rates[destination_rate] * dm[time_index]
                - transition_coordinate_delta[time_index]
            )
            offsets, position_weights = position_kernel_probabilities(
                mean_shift,
                step,
                sig_p,
            )
            position_log_weights = np.log(position_weights)
            for source_position in range(position_count):
                best = negative
                for kernel_index in range(5):
                    destination_position = (
                        source_position + offsets[kernel_index]
                    )
                    if 0 <= destination_position < position_count:
                        value = (
                            position_log_weights[kernel_index]
                            + emission_lambda
                            * emission_ll[
                                time_index,
                                destination_position,
                            ]
                            + beta_next[destination_position, destination_rate]
                        )
                        if value > best:
                            best = value
                if best > negative / 2:
                    total = 0.0
                    for kernel_index in range(5):
                        destination_position = (
                            source_position + offsets[kernel_index]
                        )
                        if 0 <= destination_position < position_count:
                            total += np.exp(
                                position_log_weights[kernel_index]
                                + emission_lambda
                                * emission_ll[
                                    time_index,
                                    destination_position,
                                ]
                                + beta_next[
                                    destination_position,
                                    destination_rate,
                                ]
                                - best
                            )
                    beta_position[source_position, destination_rate] = np.float32(
                        best + np.log(total)
                    )
                else:
                    beta_position[source_position, destination_rate] = negative

        for source_position in range(position_count):
            for source_rate in range(rate_count):
                best = negative
                first_destination = max(source_rate - 1, 0)
                last_destination = min(source_rate + 1, rate_count - 1)
                for destination_rate in range(
                    first_destination,
                    last_destination + 1,
                ):
                    value = (
                        rate_log_kernel[
                            source_rate,
                            destination_rate - source_rate + 1,
                        ]
                        + beta_position[source_position, destination_rate]
                    )
                    if value > best:
                        best = value
                if best > negative / 2:
                    total = 0.0
                    for destination_rate in range(
                        first_destination,
                        last_destination + 1,
                    ):
                        total += np.exp(
                            rate_log_kernel[
                                source_rate,
                                destination_rate - source_rate + 1,
                            ]
                            + beta_position[source_position, destination_rate]
                            - best
                        )
                    beta_current[source_position, source_rate] = np.float32(
                        best + np.log(total)
                    )
                else:
                    beta_current[source_position, source_rate] = negative

        values = alpha[time_index - 1] + beta_current
        best = np.max(values)
        total = 0.0
        for position_index in range(position_count):
            for rate_index in range(rate_count):
                total += np.exp(values[position_index, rate_index] - best)
        for position_index in range(position_count):
            for rate_index in range(rate_count):
                probability = (
                    np.exp(values[position_index, rate_index] - best) / total
                )
                posterior_position[time_index - 1, position_index] += probability
                posterior_rate[time_index - 1, rate_index] += probability
                beta_next[position_index, rate_index] = beta_current[
                    position_index, rate_index
                ]

    maximum_normalization_error = 0.0
    for time_index in range(time_count):
        position_total = np.sum(posterior_position[time_index])
        rate_total = np.sum(posterior_rate[time_index])
        for position_index in range(position_count):
            posterior_position[time_index, position_index] /= position_total
        for rate_index in range(rate_count):
            posterior_rate[time_index, rate_index] /= rate_total
        position_total = np.sum(posterior_position[time_index])
        rate_total = np.sum(posterior_rate[time_index])
        maximum_normalization_error = max(
            maximum_normalization_error,
            abs(position_total - 1.0),
            abs(rate_total - 1.0),
        )
    return (
        posterior_position,
        posterior_rate,
        log_likelihood,
        maximum_normalization_error,
    )


def transition_quantization_ledger(
    *,
    dm: np.ndarray,
    dz: np.ndarray,
    rates: np.ndarray,
    posterior_rate: np.ndarray,
    step: float,
    sig_p: float,
    row_idx: np.ndarray,
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for time_index in range(len(dm)):
        parent_weighted_abs = 0.0
        candidate_weighted_abs = 0.0
        parent_weighted_signed = 0.0
        candidate_weighted_signed = 0.0
        for rate_index, rate in enumerate(rates):
            weight = float(posterior_rate[time_index, rate_index])
            parent_mean = float(rate * dm[time_index] - dz[time_index])
            candidate_mean = float(rate * dm[time_index])
            parent_offsets, parent_weights = position_kernel_probabilities(
                parent_mean,
                step,
                sig_p,
            )
            candidate_offsets, candidate_weights = position_kernel_probabilities(
                candidate_mean,
                step,
                sig_p,
            )
            parent_expected = float(
                np.sum(parent_offsets.astype(np.float64) * step * parent_weights)
            )
            candidate_expected = float(
                np.sum(
                    candidate_offsets.astype(np.float64)
                    * step
                    * candidate_weights
                )
            )
            parent_bias = parent_expected - parent_mean
            candidate_bias = candidate_expected - candidate_mean
            parent_weighted_abs += weight * abs(parent_bias)
            candidate_weighted_abs += weight * abs(candidate_bias)
            parent_weighted_signed += weight * parent_bias
            candidate_weighted_signed += weight * candidate_bias
        rows.append(
            {
                "row_idx": int(row_idx[time_index]),
                "delta_md": float(dm[time_index]),
                "delta_z": float(dz[time_index]),
                "parent_posterior_weighted_abs_quantization_bias_ft": (
                    parent_weighted_abs
                ),
                "candidate_posterior_weighted_abs_quantization_bias_ft": (
                    candidate_weighted_abs
                ),
                "parent_posterior_weighted_signed_quantization_bias_ft": (
                    parent_weighted_signed
                ),
                "candidate_posterior_weighted_signed_quantization_bias_ft": (
                    candidate_weighted_signed
                ),
                "nontrivial_z_phase": bool(
                    abs(dz[time_index] / step - round(dz[time_index] / step))
                    > 1.0e-12
                ),
            }
        )
    return pd.DataFrame(rows)


def transition_stochastic_contract(
    *,
    dm: np.ndarray,
    rates: np.ndarray,
    step: float,
    sig_r: float,
    sig_p: float,
    momentum: float,
) -> dict[str, Any]:
    maximum_rate_error = 0.0
    maximum_position_error = 0.0
    for delta_md in np.asarray(dm, dtype=np.float64):
        rate_kernel = rate_kernel_probabilities(
            rates,
            float(delta_md),
            sig_r,
            momentum,
        )
        maximum_rate_error = max(
            maximum_rate_error,
            float(np.max(np.abs(rate_kernel.sum(axis=1) - 1.0))),
        )
        for rate in rates:
            _, weights = position_kernel_probabilities(
                float(rate * delta_md),
                step,
                sig_p,
            )
            maximum_position_error = max(
                maximum_position_error,
                abs(float(weights.sum()) - 1.0),
            )
    return {
        "rate_kernel_row_sum_max_error": maximum_rate_error,
        "position_kernel_row_sum_max_error": maximum_position_error,
        "transition_row_sum_max_error": max(
            maximum_rate_error,
            maximum_position_error,
        ),
    }


def run_fixed_u_hmm(
    prepared: Mapping[str, Any],
    hmm: Mapping[str, Any],
) -> dict[str, Any]:
    started = time.perf_counter()
    posterior_position, posterior_rate, log_likelihood, normalization_error = (
        _hmm2_fb_fixed_lattice(
            np.asarray(prepared["emission_ll"], dtype=np.float32),
            np.asarray(prepared["dm"], dtype=np.float64),
            np.zeros_like(np.asarray(prepared["dz"], dtype=np.float64)),
            float(hmm["position_grid_step_ft"]),
            np.asarray(prepared["rates"], dtype=np.float64),
            float(hmm["sig_r"]),
            float(hmm["sig_p"]),
            float(prepared["start_p"]),
            float(hmm["start_sigma_ft"]),
            float(prepared["r0"]),
            float(hmm["initial_rate_sigma"]),
            float(hmm["emission_lambda"]),
            float(hmm["momentum"]),
        )
    )
    u_grid = np.asarray(prepared["u_grid"], dtype=np.float64)
    z = np.asarray(prepared["z"], dtype=np.float64)
    rates = np.asarray(prepared["rates"], dtype=np.float64)
    mean_u = posterior_position @ u_grid
    mean_tvt = mean_u - z
    variance_u = np.sum(
        posterior_position * (u_grid[None, :] - mean_u[:, None]) ** 2,
        axis=1,
    )
    std_tvt = np.sqrt(np.maximum(variance_u, 0.0))
    rate_mean = posterior_rate @ rates
    rate_variance = posterior_rate @ (rates**2) - rate_mean**2
    rate_std = np.sqrt(np.maximum(rate_variance, 0.0))
    rate_edge_mass = posterior_rate[:, 0] + posterior_rate[:, -1]
    readout_identity = float(
        np.max(np.abs(mean_tvt - (mean_u - z)))
    )
    quantization = transition_quantization_ledger(
        dm=np.asarray(prepared["dm"], dtype=np.float64),
        dz=np.asarray(prepared["dz"], dtype=np.float64),
        rates=rates,
        posterior_rate=posterior_rate,
        step=float(hmm["position_grid_step_ft"]),
        sig_p=float(hmm["sig_p"]),
        row_idx=np.asarray(prepared["eval_index"], dtype=np.int64),
    )
    stochastic = transition_stochastic_contract(
        dm=np.asarray(prepared["dm"], dtype=np.float64),
        rates=rates,
        step=float(hmm["position_grid_step_ft"]),
        sig_r=float(hmm["sig_r"]),
        sig_p=float(hmm["sig_p"]),
        momentum=float(hmm["momentum"]),
    )
    prediction_sha = array_bundle_sha256(
        row_idx=np.asarray(prepared["eval_index"], dtype=np.int64),
        mean_tvt=np.asarray(mean_tvt, dtype=np.float32),
        std_tvt=np.asarray(std_tvt, dtype=np.float32),
    )
    rate_sha = array_bundle_sha256(
        row_idx=np.asarray(prepared["eval_index"], dtype=np.int64),
        rate_mean=np.asarray(rate_mean, dtype=np.float32),
        rate_std=np.asarray(rate_std, dtype=np.float32),
        rate_edge_mass=np.asarray(rate_edge_mass, dtype=np.float32),
    )
    transition_sha = logical_frame_sha256(quantization)
    return {
        "posterior_position": posterior_position,
        "posterior_rate": posterior_rate,
        "mean_u": mean_u,
        "mean_tvt": mean_tvt,
        "std_tvt": std_tvt,
        "rate_mean": rate_mean,
        "rate_std": rate_std,
        "rate_edge_mass": rate_edge_mass,
        "log_likelihood": float(log_likelihood),
        "posterior_normalization_max_error": float(normalization_error),
        "readout_identity_max_abs_ft": readout_identity,
        "quantization_ledger": quantization,
        "transition_row_sum_max_error": stochastic[
            "transition_row_sum_max_error"
        ],
        "prediction_sha256": prediction_sha,
        "rate_readout_sha256": rate_sha,
        "transition_ledger_sha256": transition_sha,
        "elapsed_seconds": float(time.perf_counter() - started),
    }

## 6. Numerical contracts and target-free prediction freeze

Four independent contracts are evaluated before any suffix truth, role,
fold, persistent episode, or error is opened:

- coordinate and emission identities on every fixed32 well;
- constant-Z parent/candidate parity;
- an exhaustive small-path reference for the generic joint HMM;
- transition/posterior normalization.

The candidate prediction, posterior rate readout, and transition
quantization ledger receive content SHAs before the guarded truth-late phase.

In [ ]:
def _dense_transition_matrix(
    *,
    dm: float,
    coordinate_delta: float,
    step: float,
    rates: np.ndarray,
    sig_r: float,
    sig_p: float,
    momentum: float,
    position_count: int,
) -> np.ndarray:
    rate_count = len(rates)
    state_count = position_count * rate_count
    transition = np.zeros((state_count, state_count), dtype=np.float64)
    rate_kernel = rate_kernel_probabilities(rates, dm, sig_r, momentum)
    for source_position in range(position_count):
        for source_rate in range(rate_count):
            source_state = source_position * rate_count + source_rate
            first_destination = max(source_rate - 1, 0)
            last_destination = min(source_rate + 1, rate_count - 1)
            for destination_rate in range(
                first_destination,
                last_destination + 1,
            ):
                rate_probability = rate_kernel[
                    source_rate,
                    destination_rate - source_rate + 1,
                ]
                mean_shift = (
                    rates[destination_rate] * dm - coordinate_delta
                )
                offsets, position_weights = position_kernel_probabilities(
                    mean_shift,
                    step,
                    sig_p,
                )
                for kernel_index, offset in enumerate(offsets):
                    destination_position = source_position + int(offset)
                    if 0 <= destination_position < position_count:
                        destination_state = (
                            destination_position * rate_count
                            + destination_rate
                        )
                        transition[source_state, destination_state] += (
                            rate_probability * position_weights[kernel_index]
                        )
    return transition


def exhaustive_small_path_reference(
    emission_ll: np.ndarray,
    dm: np.ndarray,
    coordinate_delta: np.ndarray,
    step: float,
    rates: np.ndarray,
    sig_r: float,
    sig_p: float,
    start_p: float,
    start_sig: float,
    r0: float,
    r0_sig: float,
    emission_lambda: float,
    momentum: float,
) -> tuple[np.ndarray, np.ndarray, float]:
    """Enumerate every initial/suffix state path for a deliberately tiny HMM."""
    emission_ll = np.asarray(emission_ll, dtype=np.float64)
    dm = np.asarray(dm, dtype=np.float64)
    coordinate_delta = np.asarray(coordinate_delta, dtype=np.float64)
    rates = np.asarray(rates, dtype=np.float64)
    time_count, position_count = emission_ll.shape
    rate_count = len(rates)
    state_count = position_count * rate_count
    if time_count > 3 or state_count > 9:
        raise ValueError("exhaustive reference is intentionally limited to tiny HMMs")

    initial = np.zeros(state_count, dtype=np.float64)
    for position_index in range(position_count):
        delta_position = (position_index - start_p) * step
        position_logp = -0.5 * (delta_position / start_sig) ** 2
        if position_logp < -60.0:
            continue
        for rate_index, rate in enumerate(rates):
            delta_rate = (rate - r0) / r0_sig
            initial[position_index * rate_count + rate_index] = np.exp(
                position_logp - 0.5 * delta_rate * delta_rate
            )
    transitions = [
        _dense_transition_matrix(
            dm=float(dm[time_index]),
            coordinate_delta=float(coordinate_delta[time_index]),
            step=step,
            rates=rates,
            sig_r=sig_r,
            sig_p=sig_p,
            momentum=momentum,
            position_count=position_count,
        )
        for time_index in range(time_count)
    ]
    emission_probability = np.empty((time_count, state_count), dtype=np.float64)
    for time_index in range(time_count):
        for position_index in range(position_count):
            value = np.exp(emission_lambda * emission_ll[time_index, position_index])
            start = position_index * rate_count
            emission_probability[
                time_index,
                start : start + rate_count,
            ] = value

    path_states: list[tuple[int, ...]] = []
    path_weights: list[float] = []

    def extend_path(
        time_index: int,
        previous_state: int,
        path: tuple[int, ...],
        weight: float,
    ) -> None:
        if time_index == time_count:
            path_states.append(path)
            path_weights.append(weight)
            return
        transition = transitions[time_index]
        for destination_state in range(state_count):
            next_weight = (
                weight
                * transition[previous_state, destination_state]
                * emission_probability[time_index, destination_state]
            )
            if next_weight > 0.0:
                extend_path(
                    time_index + 1,
                    destination_state,
                    (*path, destination_state),
                    next_weight,
                )

    for initial_state in range(state_count):
        if initial[initial_state] > 0.0:
            extend_path(0, initial_state, (), float(initial[initial_state]))
    weights = np.asarray(path_weights, dtype=np.float64)
    total = float(weights.sum())
    if not np.isfinite(total) or total <= 0.0:
        raise RuntimeError("exhaustive reference has zero or non-finite mass")
    normalized = weights / total
    posterior_position = np.zeros((time_count, position_count), dtype=np.float64)
    posterior_rate = np.zeros((time_count, rate_count), dtype=np.float64)
    for probability, path in zip(normalized, path_states, strict=True):
        for time_index, state in enumerate(path):
            position_index = state // rate_count
            rate_index = state % rate_count
            posterior_position[time_index, position_index] += probability
            posterior_rate[time_index, rate_index] += probability
    return posterior_position, posterior_rate, float(np.log(total))


def brute_force_small_reference_contract(
    hmm: Mapping[str, Any],
) -> dict[str, Any]:
    emission_ll = np.asarray(
        [
            [-0.20, -0.01, -0.40],
            [-0.35, -0.05, -0.10],
            [-0.60, -0.15, -0.02],
        ],
        dtype=np.float32,
    )
    dm = np.asarray([1.0, 1.25, 0.75], dtype=np.float64)
    coordinate_delta = np.asarray([0.08, -0.04, 0.11], dtype=np.float64)
    rates = np.asarray([-0.02, 0.0, 0.02], dtype=np.float64)
    parameters = {
        "step": float(hmm["position_grid_step_ft"]),
        "sig_r": float(hmm["sig_r"]),
        "sig_p": float(hmm["sig_p"]),
        "start_p": 1.1,
        "start_sig": float(hmm["start_sigma_ft"]),
        "r0": 0.0,
        "r0_sig": float(hmm["initial_rate_sigma"]),
        "emission_lambda": float(hmm["emission_lambda"]),
        "momentum": float(hmm["momentum"]),
    }
    observed_position, observed_rate, observed_loglik, observed_normalization = (
        _hmm2_fb_fixed_lattice(
            emission_ll,
            dm,
            coordinate_delta,
            parameters["step"],
            rates,
            parameters["sig_r"],
            parameters["sig_p"],
            parameters["start_p"],
            parameters["start_sig"],
            parameters["r0"],
            parameters["r0_sig"],
            parameters["emission_lambda"],
            parameters["momentum"],
        )
    )
    reference_position, reference_rate, reference_loglik = (
        exhaustive_small_path_reference(
            emission_ll,
            dm,
            coordinate_delta,
            parameters["step"],
            rates,
            parameters["sig_r"],
            parameters["sig_p"],
            parameters["start_p"],
            parameters["start_sig"],
            parameters["r0"],
            parameters["r0_sig"],
            parameters["emission_lambda"],
            parameters["momentum"],
        )
    )
    position_diff = float(
        np.max(np.abs(observed_position - reference_position))
    )
    rate_diff = float(np.max(np.abs(observed_rate - reference_rate)))
    loglik_diff = abs(float(observed_loglik) - float(reference_loglik))
    maximum = max(position_diff, rate_diff, loglik_diff)
    return {
        "position_posterior_max_abs": position_diff,
        "rate_posterior_max_abs": rate_diff,
        "log_likelihood_abs": loglik_diff,
        "maximum_abs": maximum,
        "posterior_normalization_max_error": float(observed_normalization),
        "pass": bool(maximum <= 1.0e-6),
    }


def constant_z_parent_parity_contract(
    hmm: Mapping[str, Any],
) -> dict[str, Any]:
    rows = 7
    positions = 11
    z_constant = 8123.75
    parent_grid = 11_950.0 + np.arange(positions, dtype=np.float64) * float(
        hmm["position_grid_step_ft"]
    )
    u_grid = parent_grid + z_constant
    emission_ll = np.vstack(
        [
            -0.5
            * (
                (
                    np.linspace(-1.0, 1.0, positions)
                    - 0.25 * np.sin(row / 2.0)
                )
                / 0.45
            )
            ** 2
            for row in range(rows)
        ]
    ).astype(np.float32)
    dm = 1.0 + (np.arange(rows, dtype=np.float64) % 3) * 0.25
    dz = np.zeros(rows, dtype=np.float64)
    rates = np.linspace(-0.04, 0.04, 9, dtype=np.float64)
    common = (
        emission_ll,
        dm,
        dz,
        float(hmm["position_grid_step_ft"]),
        rates,
        float(hmm["sig_r"]),
        float(hmm["sig_p"]),
        5.2,
        float(hmm["start_sigma_ft"]),
        0.01,
        float(hmm["initial_rate_sigma"]),
        float(hmm["emission_lambda"]),
        float(hmm["momentum"]),
    )
    parent_position, parent_rate, parent_loglik, parent_norm = (
        _hmm2_fb_fixed_lattice(*common)
    )
    candidate_position, candidate_rate, candidate_loglik, candidate_norm = (
        _hmm2_fb_fixed_lattice(*common)
    )
    parent_prediction = parent_position @ parent_grid
    candidate_prediction = candidate_position @ u_grid - z_constant
    prediction_diff = float(
        np.max(np.abs(parent_prediction - candidate_prediction))
    )
    position_diff = float(
        np.max(np.abs(parent_position - candidate_position))
    )
    rate_diff = float(np.max(np.abs(parent_rate - candidate_rate)))
    loglik_diff = abs(float(parent_loglik) - float(candidate_loglik))
    maximum = max(prediction_diff, position_diff, rate_diff, loglik_diff)
    return {
        "prediction_max_abs_ft": prediction_diff,
        "position_posterior_max_abs": position_diff,
        "rate_posterior_max_abs": rate_diff,
        "log_likelihood_abs": loglik_diff,
        "posterior_normalization_max_error": max(
            float(parent_norm),
            float(candidate_norm),
        ),
        "pass": bool(maximum <= 1.0e-6),
    }


@dataclass
class FrozenWell:
    well: str
    eval_id: np.ndarray
    row_idx: np.ndarray
    raw_gr_missing: np.ndarray
    parent_prediction: np.ndarray
    candidate_prediction: np.ndarray
    candidate_posterior_std: np.ndarray
    candidate_rate_mean: np.ndarray
    candidate_rate_std: np.ndarray
    candidate_rate_edge_mass: np.ndarray
    quantization_ledger: pd.DataFrame
    coordinate_contract: dict[str, Any]
    prediction_sha256: str
    rate_readout_sha256: str
    transition_ledger_sha256: str
    log_likelihood: float
    transition_row_sum_max_error: float
    posterior_normalization_max_error: float
    readout_identity_max_abs_ft: float
    elapsed_seconds: float
    last_known_tvt: float
    last_known_md: float
    last_known_z: float
    prefix_rows: int
    role: str | None = None
    fold: int | None = None


def freeze_target_free_well(
    *,
    well: str,
    expected_prefix_rows: int,
    expected_suffix_rows: int,
    parent_rows: pd.DataFrame,
    raw_dir: Path,
    hmm: Mapping[str, Any],
    ledger: LeakageLedger,
) -> FrozenWell:
    horizontal, typewell = load_target_free_well(well, raw_dir, ledger)
    prepared = prepare_hmm_inputs(horizontal, typewell, hmm)
    if int(prepared["prefix_rows"]) != int(expected_prefix_rows):
        raise ValueError(f"{well}: prefix row count changed")
    if len(prepared["eval_index"]) != int(expected_suffix_rows):
        raise ValueError(f"{well}: suffix row count changed")
    expected_ids = parent_cache_ids_for_rows(
        well,
        np.asarray(prepared["eval_index"], dtype=np.int64),
    )
    aligned_parent = parent_rows.sort_values("row_idx", kind="mergesort")
    if not np.array_equal(
        aligned_parent["id"].astype(str).to_numpy(),
        expected_ids,
    ):
        raise ValueError(f"{well}: saved parent row identity changed")
    coordinate = coordinate_contract_from_prepared(prepared)
    result = run_fixed_u_hmm(prepared, hmm)
    quantization = result["quantization_ledger"].copy()
    quantization.insert(0, "well", well)
    frozen = FrozenWell(
        well=well,
        eval_id=expected_ids,
        row_idx=np.asarray(prepared["eval_index"], dtype=np.int64),
        raw_gr_missing=np.asarray(prepared["raw_gr_missing"], dtype=bool),
        parent_prediction=aligned_parent["parent_prediction"].to_numpy(np.float64),
        candidate_prediction=np.asarray(result["mean_tvt"], dtype=np.float64),
        candidate_posterior_std=np.asarray(result["std_tvt"], dtype=np.float64),
        candidate_rate_mean=np.asarray(result["rate_mean"], dtype=np.float64),
        candidate_rate_std=np.asarray(result["rate_std"], dtype=np.float64),
        candidate_rate_edge_mass=np.asarray(
            result["rate_edge_mass"],
            dtype=np.float64,
        ),
        quantization_ledger=quantization,
        coordinate_contract=coordinate,
        prediction_sha256=str(result["prediction_sha256"]),
        rate_readout_sha256=str(result["rate_readout_sha256"]),
        transition_ledger_sha256=str(result["transition_ledger_sha256"]),
        log_likelihood=float(result["log_likelihood"]),
        transition_row_sum_max_error=float(
            result["transition_row_sum_max_error"]
        ),
        posterior_normalization_max_error=float(
            result["posterior_normalization_max_error"]
        ),
        readout_identity_max_abs_ft=float(
            result["readout_identity_max_abs_ft"]
        ),
        elapsed_seconds=float(result["elapsed_seconds"]),
        last_known_tvt=float(prepared["last_known_tvt"]),
        last_known_md=float(prepared["last_known_md"]),
        last_known_z=float(prepared["last_known_z"]),
        prefix_rows=int(prepared["prefix_rows"]),
    )
    ledger.freeze(well)
    return frozen


def attach_scope_identity(
    frozen_wells: Sequence[FrozenWell],
    identity: pd.DataFrame,
) -> None:
    by_well = identity.set_index("well")
    if set(by_well.index) != {item.well for item in frozen_wells}:
        raise ValueError("fixed32 identity/prediction wells differ")
    for item in frozen_wells:
        row = by_well.loc[item.well]
        item.role = str(row["role"])
        item.fold = int(row["fold"])


def prediction_frame(frozen_wells: Sequence[FrozenWell]) -> pd.DataFrame:
    pieces: list[pd.DataFrame] = []
    for item in frozen_wells:
        pieces.append(
            pd.DataFrame(
                {
                    "id": item.eval_id,
                    "well": item.well,
                    "row_idx": item.row_idx,
                    "parent_exp209_tvt": item.parent_prediction,
                    "candidate_fixed_u_tvt": item.candidate_prediction,
                    "candidate_posterior_std_tvt": (
                        item.candidate_posterior_std
                    ),
                }
            )
        )
    return pd.concat(pieces, ignore_index=True).sort_values(
        ["well", "row_idx"],
        kind="mergesort",
    )


def rate_readout_frame(frozen_wells: Sequence[FrozenWell]) -> pd.DataFrame:
    pieces: list[pd.DataFrame] = []
    for item in frozen_wells:
        pieces.append(
            pd.DataFrame(
                {
                    "well": item.well,
                    "row_idx": item.row_idx,
                    "candidate_rate_mean": item.candidate_rate_mean,
                    "candidate_rate_std": item.candidate_rate_std,
                    "candidate_rate_edge_mass": item.candidate_rate_edge_mass,
                }
            )
        )
    return pd.concat(pieces, ignore_index=True).sort_values(
        ["well", "row_idx"],
        kind="mergesort",
    )


def quantization_frame(frozen_wells: Sequence[FrozenWell]) -> pd.DataFrame:
    return pd.concat(
        [item.quantization_ledger for item in frozen_wells],
        ignore_index=True,
    ).sort_values(["well", "row_idx"], kind="mergesort")


def coordinate_contract_summary(
    frozen_wells: Sequence[FrozenWell],
) -> dict[str, Any]:
    return {
        "tvt_equals_u_minus_z_max_abs_ft": max(
            item.coordinate_contract["tvt_equals_u_minus_z_max_abs_ft"]
            for item in frozen_wells
        ),
        "transition_coordinate_identity_max_abs_ft": max(
            item.coordinate_contract[
                "transition_coordinate_identity_max_abs_ft"
            ]
            for item in frozen_wells
        ),
        "emission_coordinate_identity_max_abs": max(
            item.coordinate_contract["emission_coordinate_identity_max_abs"]
            for item in frozen_wells
        ),
        "readout_identity_max_abs_ft": max(
            item.readout_identity_max_abs_ft for item in frozen_wells
        ),
        "per_well_sha256": {
            item.well: item.coordinate_contract["sha256"]
            for item in frozen_wells
        },
    }

## 7. Truth-late persistent-episode and safety readout

Suffix TVT, role/fold identity, persistent episodes, exp408 cause labels, and
all error metrics are opened only after all candidate predictions, posterior
rate readouts, coordinate contracts, and transition ledgers are frozen.

In [ ]:
def load_truth_after_all_freeze(
    frozen: FrozenWell,
    raw_dir: Path,
    ledger: LeakageLedger,
) -> pd.DataFrame:
    frame = pd.read_csv(
        raw_dir / f"{frozen.well}__horizontal_well.csv",
        usecols=["MD", "Z", "TVT", "TVT_input"],
    )
    suffix = frame.loc[frame["TVT_input"].isna()].copy()
    ledger.record_truth_late(len(suffix))
    if not np.array_equal(suffix.index.to_numpy(np.int64), frozen.row_idx):
        raise ValueError(f"{frozen.well}: truth row index changed after freeze")
    reconstructed_ids = parent_cache_ids_for_rows(
        frozen.well,
        suffix.index.to_numpy(np.int64),
    )
    if not np.array_equal(reconstructed_ids, frozen.eval_id):
        raise ValueError(f"{frozen.well}: truth id changed after freeze")
    suffix["id"] = reconstructed_ids
    return suffix.reset_index(names="row_idx")


def well_truth_late_metrics(
    frozen: FrozenWell,
    truth: pd.DataFrame,
) -> dict[str, Any]:
    if frozen.role is None or frozen.fold is None:
        raise RuntimeError("role/fold identity was not attached after freeze")
    actual = truth["TVT"].to_numpy(np.float64)
    parent_error = frozen.parent_prediction - actual
    candidate_error = frozen.candidate_prediction - actual
    parent_rmse = float(np.sqrt(np.mean(parent_error**2)))
    candidate_rmse = float(np.sqrt(np.mean(candidate_error**2)))
    return {
        "well": frozen.well,
        "role": frozen.role,
        "fold": frozen.fold,
        "rows": len(actual),
        "parent_rmse_ft": parent_rmse,
        "candidate_rmse_ft": candidate_rmse,
        "rmse_delta_vs_parent_ft": candidate_rmse - parent_rmse,
        "improved_vs_parent": candidate_rmse < parent_rmse,
        "raw_gr_missing_fraction": float(np.mean(frozen.raw_gr_missing)),
        "prediction_sha256": frozen.prediction_sha256,
        "rate_readout_sha256": frozen.rate_readout_sha256,
        "transition_ledger_sha256": frozen.transition_ledger_sha256,
        "hmm_seconds": frozen.elapsed_seconds,
    }


def load_persistent_episodes_after_all_freeze(
    config: Mapping[str, Any],
    selected_persistent: set[str],
    ledger: LeakageLedger,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    spec = get_nested(config, "data.persistent_episodes")
    path = resolve_bootstrap_asset(str(spec["filename"]), str(spec["local"]))
    observed = sha256_file(path)
    if observed != str(spec["expected_sha256"]):
        raise ValueError(f"persistent episode SHA changed: {observed}")
    frame = pd.read_csv(path, dtype={"well": str, "episode_id": str})
    frame = frame.loc[frame["well"].isin(selected_persistent)].copy()
    ledger.record_episode_late(len(frame))
    required = {
        "episode_id",
        "well",
        "start_row_idx",
        "end_row_idx_exclusive",
        "start_suffix_offset",
        "rows",
    }
    if not required.issubset(frame.columns):
        raise ValueError("persistent episode schema changed")
    if frame.empty or frame["well"].nunique() != len(selected_persistent):
        raise ValueError("selected persistent wells are missing episode rows")

    cause_spec = get_nested(config, "data.exp408_episode_causes")
    cause_path = resolve_bootstrap_asset(
        str(cause_spec["filename"]),
        str(cause_spec["local"]),
    )
    cause_sha = sha256_file(cause_path)
    if cause_sha != str(cause_spec["expected_sha256"]):
        raise ValueError(f"exp408 episode-cause SHA changed: {cause_sha}")
    causes = pd.read_csv(
        cause_path,
        usecols=["episode_id", "well", "fold", "cause"],
        dtype={"episode_id": str, "well": str},
    )
    causes = causes.loc[causes["well"].isin(selected_persistent)].copy()
    ledger.record_episode_late(len(causes))
    if causes["episode_id"].duplicated().any():
        raise ValueError("exp408 episode-cause identity is not unique")
    frame = frame.merge(
        causes,
        on=["episode_id", "well"],
        how="left",
        validate="one_to_one",
    )
    if frame["cause"].isna().any():
        raise ValueError("exp408 cause is missing for selected persistent episodes")
    return frame.sort_values(
        ["well", "start_suffix_offset"],
        kind="mergesort",
    ).reset_index(drop=True), {
        "path": str(path),
        "sha256": observed,
        "cause_path": str(cause_path),
        "cause_sha256": cause_sha,
        "selected_rows": len(frame),
        "selected_wells": frame["well"].nunique(),
    }


def episode_truth_late_readout(
    episodes: pd.DataFrame,
    frozen_by_well: Mapping[str, FrozenWell],
    truth_by_well: Mapping[str, pd.DataFrame],
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for episode in episodes.itertuples(index=False):
        frozen = frozen_by_well[str(episode.well)]
        if frozen.fold is None or int(episode.fold) != int(frozen.fold):
            raise ValueError(f"{episode.episode_id}: exp408/manifest fold changed")
        truth = truth_by_well[str(episode.well)]
        row_idx = truth["row_idx"].to_numpy(np.int64)
        mask = (row_idx >= int(episode.start_row_idx)) & (
            row_idx < int(episode.end_row_idx_exclusive)
        )
        offsets = np.flatnonzero(mask)
        if len(offsets) != int(episode.rows):
            raise ValueError(f"{episode.episode_id}: episode row coverage changed")
        actual = truth["TVT"].to_numpy(np.float64)[offsets]
        parent_error = frozen.parent_prediction[offsets] - actual
        candidate_error = frozen.candidate_prediction[offsets] - actual
        parent_sse = float(np.sum(parent_error**2))
        candidate_sse = float(np.sum(candidate_error**2))
        rows.append(
            {
                "episode_id": str(episode.episode_id),
                "well": str(episode.well),
                "fold": int(frozen.fold),
                "cause": str(episode.cause),
                "rows": len(offsets),
                "start_row_idx": int(episode.start_row_idx),
                "end_row_idx_exclusive": int(episode.end_row_idx_exclusive),
                "parent_sse": parent_sse,
                "candidate_sse": candidate_sse,
                "candidate_sse_reduction_vs_parent": (
                    1.0 - candidate_sse / parent_sse
                    if parent_sse > 0.0
                    else math.nan
                ),
            }
        )
    return pd.DataFrame(rows).sort_values(
        ["fold", "well", "start_row_idx"],
        kind="mergesort",
    )

## 8. Stage 0 gates, generated artifacts, and metrics

The fixed32 result is a mechanism preflight, not CV or promotion evidence.
Every technical and mechanism gate is an AND condition. Any failure closes
this branch without changing the U-grid anchor/phase/step/band, position or
rate noise, emission, selector, or blend on the same fixed32 sample.

In [ ]:
def fraction(numerator: int | float, denominator: int | float) -> float:
    return float(numerator / denominator) if denominator else math.nan


def pooled_rmse_from_well_rows(
    frame: pd.DataFrame,
    column: str,
) -> float:
    weights = frame["rows"].to_numpy(np.float64)
    values = frame[column].to_numpy(np.float64)
    return float(np.sqrt(np.average(values**2, weights=weights)))


def finite_readout_coverage(
    frozen_wells: Sequence[FrozenWell],
) -> float:
    finite = 0
    total = 0
    attributes = (
        "candidate_prediction",
        "candidate_posterior_std",
        "candidate_rate_mean",
        "candidate_rate_std",
        "candidate_rate_edge_mass",
    )
    for item in frozen_wells:
        for attribute in attributes:
            values = np.asarray(getattr(item, attribute), dtype=np.float64)
            finite += int(np.isfinite(values).sum())
            total += int(values.size)
    return fraction(finite, total)


def evaluate_mechanism_gates(
    *,
    config: Mapping[str, Any],
    quantization: pd.DataFrame,
    episode_readout: pd.DataFrame,
    well_metrics: pd.DataFrame,
) -> dict[str, Any]:
    mechanism = get_nested(config, "gates.stage0_fixed32.mechanism")
    parent_quantization = float(
        quantization[
            "parent_posterior_weighted_abs_quantization_bias_ft"
        ].sum()
    )
    candidate_quantization = float(
        quantization[
            "candidate_posterior_weighted_abs_quantization_bias_ft"
        ].sum()
    )
    quantization_reduction = (
        1.0 - candidate_quantization / parent_quantization
        if parent_quantization > 0.0
        else math.nan
    )

    parent_episode_sse = float(episode_readout["parent_sse"].sum())
    candidate_episode_sse = float(episode_readout["candidate_sse"].sum())
    persistent_reduction = (
        1.0 - candidate_episode_sse / parent_episode_sse
        if parent_episode_sse > 0.0
        else math.nan
    )
    forward_cause = str(
        get_nested(config, "data.exp408_episode_causes.forward_cause")
    )
    forward = episode_readout.loc[
        episode_readout["cause"].eq(forward_cause)
    ]
    forward_parent_sse = float(forward["parent_sse"].sum())
    forward_candidate_sse = float(forward["candidate_sse"].sum())
    forward_reduction = (
        1.0 - forward_candidate_sse / forward_parent_sse
        if forward_parent_sse > 0.0
        else math.nan
    )

    persistent_wells = well_metrics.loc[
        well_metrics["role"].eq("persistent")
    ]
    control_wells = well_metrics.loc[well_metrics["role"].eq("control")]
    improved_wells = int(
        persistent_wells["improved_vs_parent"].astype(bool).sum()
    )
    fold_rows: list[dict[str, Any]] = []
    for fold in range(5):
        fold_episodes = episode_readout.loc[
            episode_readout["fold"].eq(fold)
        ]
        parent_sse = float(fold_episodes["parent_sse"].sum())
        candidate_sse = float(fold_episodes["candidate_sse"].sum())
        fold_rows.append(
            {
                "fold": fold,
                "episodes": len(fold_episodes),
                "parent_sse": parent_sse,
                "candidate_sse": candidate_sse,
                "improved": bool(
                    len(fold_episodes) > 0 and candidate_sse < parent_sse
                ),
            }
        )
    improving_folds = int(sum(row["improved"] for row in fold_rows))
    control_parent_rmse = pooled_rmse_from_well_rows(
        control_wells,
        "parent_rmse_ft",
    )
    control_candidate_rmse = pooled_rmse_from_well_rows(
        control_wells,
        "candidate_rmse_ft",
    )
    control_delta = control_candidate_rmse - control_parent_rmse
    control_p95 = float(
        np.quantile(
            control_wells["rmse_delta_vs_parent_ft"].to_numpy(np.float64),
            0.95,
        )
    )
    gates = {
        "posterior_weighted_abs_quantization_bias_reduction": bool(
            math.isfinite(quantization_reduction)
            and quantization_reduction
            >= float(
                mechanism[
                    "posterior_weighted_abs_quantization_bias_reduction_min_fraction"
                ]
            )
        ),
        "forward_cause_episode_sse_reduction": bool(
            math.isfinite(forward_reduction)
            and forward_reduction
            >= float(
                mechanism[
                    "forward_cause_episode_sse_reduction_min_fraction"
                ]
            )
        ),
        "persistent_episode_sse_reduction": bool(
            math.isfinite(persistent_reduction)
            and persistent_reduction
            >= float(
                mechanism[
                    "persistent_episode_sse_reduction_min_fraction"
                ]
            )
        ),
        "persistent_improved_wells": bool(
            improved_wells
            >= int(mechanism["persistent_improved_wells_min"])
        ),
        "persistent_improving_folds": bool(
            improving_folds
            >= int(mechanism["persistent_improving_folds_min"])
        ),
        "matched_control_pooled_rmse": bool(
            control_delta
            <= float(mechanism["matched_control_pooled_rmse_delta_max_ft"])
        ),
        "matched_control_by_well_p95": bool(
            control_p95
            <= float(
                mechanism["matched_control_by_well_delta_p95_max_ft"]
            )
        ),
    }
    return {
        "gates": gates,
        "all_mechanism_gates_pass": bool(all(gates.values())),
        "diagnostics": {
            "parent_posterior_weighted_abs_quantization_bias_sum_ft": (
                parent_quantization
            ),
            "candidate_posterior_weighted_abs_quantization_bias_sum_ft": (
                candidate_quantization
            ),
            "posterior_weighted_abs_quantization_bias_reduction_fraction": (
                quantization_reduction
            ),
            "parent_persistent_episode_sse": parent_episode_sse,
            "candidate_persistent_episode_sse": candidate_episode_sse,
            "persistent_episode_sse_reduction_fraction": persistent_reduction,
            "forward_cause": forward_cause,
            "forward_cause_episodes": len(forward),
            "forward_cause_parent_sse": forward_parent_sse,
            "forward_cause_candidate_sse": forward_candidate_sse,
            "forward_cause_episode_sse_reduction_fraction": forward_reduction,
            "persistent_improved_wells": improved_wells,
            "persistent_sse_by_fold": fold_rows,
            "persistent_improving_folds": improving_folds,
            "control_parent_rmse_ft": control_parent_rmse,
            "control_candidate_rmse_ft": control_candidate_rmse,
            "control_rmse_delta_ft": control_delta,
            "control_by_well_rmse_delta_p95_ft": control_p95,
        },
    }


def evaluate_stage0_gates(
    *,
    config: Mapping[str, Any],
    identity: pd.DataFrame,
    frozen_wells: Sequence[FrozenWell],
    coordinate_contract: Mapping[str, Any],
    constant_z_contract: Mapping[str, Any],
    brute_force_contract: Mapping[str, Any],
    quantization: pd.DataFrame,
    prediction_artifact: Mapping[str, Any],
    rate_artifact: Mapping[str, Any],
    transition_artifact: Mapping[str, Any],
    episode_readout: pd.DataFrame,
    well_metrics: pd.DataFrame,
    ledger: LeakageLedger,
    elapsed_seconds: float,
) -> dict[str, Any]:
    technical_config = get_nested(config, "gates.stage0_fixed32.technical")
    maximum_transition_error = max(
        item.transition_row_sum_max_error for item in frozen_wells
    )
    maximum_posterior_error = max(
        item.posterior_normalization_max_error for item in frozen_wells
    )
    finite_coverage = finite_readout_coverage(frozen_wells)
    treatment_seconds = float(
        sum(item.elapsed_seconds for item in frozen_wells)
    )
    runtime_projection = treatment_seconds * 773.0 / 32.0
    nontrivial_phase_rows = int(
        quantization["nontrivial_z_phase"].astype(bool).sum()
    )
    nonidentity_rows = int(
        sum(
            np.count_nonzero(
                np.asarray(item.candidate_prediction, dtype=np.float32)
                != np.asarray(item.parent_prediction, dtype=np.float32)
            )
            for item in frozen_wells
        )
    )
    technical = {
        "fixed32_roles_and_unique_wells": bool(
            len(identity) == 32
            and identity["well"].nunique() == 32
            and identity["role"].value_counts().to_dict()
            == {"persistent": 16, "control": 16}
        ),
        "fixed32_fold_coverage": bool(
            set(identity["fold"].astype(int)) == {0, 1, 2, 3, 4}
        ),
        "coordinate_tvt_equals_u_minus_z": bool(
            coordinate_contract["tvt_equals_u_minus_z_max_abs_ft"]
            <= float(
                technical_config[
                    "coordinate_tvt_equals_u_minus_z_max_abs_ft"
                ]
            )
        ),
        "transition_coordinate_identity": bool(
            coordinate_contract["transition_coordinate_identity_max_abs_ft"]
            <= float(
                technical_config[
                    "transition_coordinate_identity_max_abs_ft"
                ]
            )
        ),
        "emission_coordinate_identity": bool(
            coordinate_contract["emission_coordinate_identity_max_abs"]
            <= float(
                technical_config["emission_coordinate_identity_max_abs"]
            )
        ),
        "u_tvt_readout_identity": bool(
            coordinate_contract["readout_identity_max_abs_ft"]
            <= float(
                technical_config[
                    "coordinate_tvt_equals_u_minus_z_max_abs_ft"
                ]
            )
        ),
        "constant_z_parent_parity": bool(
            constant_z_contract["pass"]
            and constant_z_contract["prediction_max_abs_ft"]
            <= float(
                technical_config[
                    "constant_z_parent_prediction_parity_max_abs_ft"
                ]
            )
        ),
        "brute_force_small_reference": bool(
            brute_force_contract["pass"]
            and brute_force_contract["maximum_abs"]
            <= float(
                technical_config[
                    "brute_force_loglik_posterior_max_abs"
                ]
            )
        ),
        "transition_row_sum": bool(
            maximum_transition_error
            <= float(technical_config["transition_row_sum_max_error"])
        ),
        "posterior_normalization": bool(
            maximum_posterior_error
            <= float(
                technical_config["posterior_normalization_max_error"]
            )
        ),
        "finite_prediction_and_diagnostic_coverage": bool(
            finite_coverage
            >= float(technical_config["finite_coverage_min"])
        ),
        "nontrivial_lattice_phase": bool(
            nontrivial_phase_rows
            >= int(technical_config["nontrivial_lattice_phase_rows_min"])
        ),
        "candidate_is_not_saved_control_copy": bool(
            nonidentity_rows
            >= int(
                technical_config[
                    "candidate_control_prediction_nonidentity_rows_min"
                ]
            )
        ),
        "truth_reads_before_all_freeze": bool(
            ledger.truth_rows_before_all_freeze
            == int(technical_config["truth_fold_episode_reads_before_freeze"])
        ),
        "role_fold_reads_before_all_freeze": bool(
            ledger.role_fold_rows_before_all_freeze
            == int(technical_config["truth_fold_episode_reads_before_freeze"])
        ),
        "episode_reads_before_all_freeze": bool(
            ledger.episode_rows_before_all_freeze
            == int(technical_config["truth_fold_episode_reads_before_freeze"])
        ),
        "prediction_readback_sha": bool(
            prediction_artifact["logical_sha256"]
            == prediction_artifact["readback_logical_sha256"]
        ),
        "rate_readout_readback_sha": bool(
            rate_artifact["logical_sha256"]
            == rate_artifact["readback_logical_sha256"]
        ),
        "transition_ledger_readback_sha": bool(
            transition_artifact["logical_sha256"]
            == transition_artifact["readback_logical_sha256"]
        ),
        "runtime_projection": bool(
            runtime_projection
            <= float(
                technical_config["projected_stage1_runtime_seconds_max"]
            )
        ),
        "peak_rss": bool(
            peak_rss_gb()
            <= float(technical_config["peak_rss_gb_max"])
        ),
    }
    mechanism = evaluate_mechanism_gates(
        config=config,
        quantization=quantization,
        episode_readout=episode_readout,
        well_metrics=well_metrics,
    )
    all_pass = bool(all(technical.values()) and mechanism["all_mechanism_gates_pass"])
    return {
        "technical": technical,
        "mechanism": mechanism,
        "diagnostics": {
            "total_wells": len(frozen_wells),
            "total_suffix_rows": int(
                sum(len(item.row_idx) for item in frozen_wells)
            ),
            "maximum_transition_row_sum_error": maximum_transition_error,
            "maximum_posterior_normalization_error": maximum_posterior_error,
            "finite_coverage": finite_coverage,
            "nontrivial_lattice_phase_rows": nontrivial_phase_rows,
            "candidate_control_prediction_nonidentity_rows": (
                nonidentity_rows
            ),
            "stage0_elapsed_seconds": float(elapsed_seconds),
            "candidate_hmm_seconds": treatment_seconds,
            "stage1_runtime_projection_seconds": runtime_projection,
            "peak_rss_gb": peak_rss_gb(),
            "truth_rows_before_all_freeze": (
                ledger.truth_rows_before_all_freeze
            ),
            "role_fold_rows_before_all_freeze": (
                ledger.role_fold_rows_before_all_freeze
            ),
            "episode_rows_before_all_freeze": (
                ledger.episode_rows_before_all_freeze
            ),
            "fixed32_is_mechanism_only_not_cv_or_promotion": True,
        },
        "stage0_all_gates_pass": all_pass,
        "stage1_eligible_for_separate_approval": all_pass,
        "fail_action": get_nested(
            config,
            "gates.stage0_fixed32.fail_action",
        ),
        "fixed32_is_cv": False,
        "fixed32_is_promotion_evidence": False,
    }


def require_kaggle_runtime() -> None:
    if KAGGLE_WORKING_ROOT.is_dir():
        return
    if os.environ.get("EXP438_ALLOW_LOCAL", "0") == "1":
        return
    raise RuntimeError("exp438 Stage 0 must run on Kaggle CPU; local execution is disabled")


def run_stage0(config: Mapping[str, Any]) -> dict[str, Any]:
    require_kaggle_runtime()
    started = time.perf_counter()
    execution_contract = validate_execution_contract(
        config,
        require_run_authorization=True,
    )
    scientific_contract = validate_scientific_contract(config)
    set_num_threads(1)
    hmm = scientific_contract["fixed_from_exp209"]
    constant_z_contract = constant_z_parent_parity_contract(hmm)
    brute_force_contract = brute_force_small_reference_contract(hmm)

    ledger = LeakageLedger(expected_wells=32)
    scope, manifest_report = load_fixed32_target_free_scope(config, ledger)
    expected_rows = int(scope["suffix_rows"].sum())
    parent, parent_report = load_saved_parent_predictions(
        config,
        set(scope["well"].astype(str)),
        expected_rows,
        ledger,
    )
    raw_dir = train_data_dir(config)

    frozen_wells: list[FrozenWell] = []
    for scope_row in scope.itertuples(index=False):
        well = str(scope_row.well)
        parent_rows = parent.loc[parent["well"].eq(well)].copy()
        frozen = freeze_target_free_well(
            well=well,
            expected_prefix_rows=int(scope_row.prefix_rows),
            expected_suffix_rows=int(scope_row.suffix_rows),
            parent_rows=parent_rows,
            raw_dir=raw_dir,
            hmm=hmm,
            ledger=ledger,
        )
        frozen_wells.append(frozen)
        print(
            json.dumps(
                {
                    "well": well,
                    "rows": len(frozen.row_idx),
                    "seconds": frozen.elapsed_seconds,
                    "prediction_sha256": frozen.prediction_sha256,
                    "transition_ledger_sha256": (
                        frozen.transition_ledger_sha256
                    ),
                },
                sort_keys=True,
            )
        )
    if not ledger.all_frozen:
        raise RuntimeError("not all fixed32 wells were frozen")

    output_dir = artifacts_dir()
    predictions = prediction_frame(frozen_wells)
    rate_readout = rate_readout_frame(frozen_wells)
    quantization = quantization_frame(frozen_wells)
    prediction_artifact = write_deterministic_gzip_csv(
        output_dir / f"{EXPERIMENT_NAME}_stage0_predictions.csv.gz",
        predictions,
    )
    rate_artifact = write_deterministic_gzip_csv(
        output_dir / f"{EXPERIMENT_NAME}_stage0_rate_readout.csv.gz",
        rate_readout,
    )
    transition_artifact = write_deterministic_gzip_csv(
        output_dir / f"{EXPERIMENT_NAME}_stage0_transition_quantization.csv.gz",
        quantization,
    )
    coordinate_contract = coordinate_contract_summary(frozen_wells)
    numerical_contract = {
        "coordinate": coordinate_contract,
        "constant_z_parent_parity": constant_z_contract,
        "brute_force_small_reference": brute_force_contract,
    }
    numerical_contract_artifact = write_json(
        output_dir / f"{EXPERIMENT_NAME}_stage0_numerical_contract.json",
        numerical_contract,
    )

    # The guarded truth-late phase begins only after all target-free SHAs exist.
    identity = load_fixed32_identity_after_all_freeze(config, ledger)
    attach_scope_identity(frozen_wells, identity)
    truth_by_well: dict[str, pd.DataFrame] = {}
    well_metric_rows: list[dict[str, Any]] = []
    for item in frozen_wells:
        truth = load_truth_after_all_freeze(item, raw_dir, ledger)
        truth_by_well[item.well] = truth
        well_metric_rows.append(well_truth_late_metrics(item, truth))
    well_metrics = pd.DataFrame(well_metric_rows).sort_values(
        ["role", "fold", "well"],
        kind="mergesort",
    )
    frozen_by_well = {item.well: item for item in frozen_wells}
    persistent_wells = set(
        identity.loc[identity["role"].eq("persistent"), "well"].astype(str)
    )
    episodes, episode_input_report = load_persistent_episodes_after_all_freeze(
        config,
        persistent_wells,
        ledger,
    )
    episode_readout = episode_truth_late_readout(
        episodes,
        frozen_by_well,
        truth_by_well,
    )
    well_artifact = write_csv(
        output_dir / f"{EXPERIMENT_NAME}_stage0_well_metrics.csv",
        well_metrics,
    )
    episode_artifact = write_csv(
        output_dir / f"{EXPERIMENT_NAME}_stage0_episode_metrics.csv",
        episode_readout,
    )
    elapsed_seconds = float(time.perf_counter() - started)
    gates = evaluate_stage0_gates(
        config=config,
        identity=identity,
        frozen_wells=frozen_wells,
        coordinate_contract=coordinate_contract,
        constant_z_contract=constant_z_contract,
        brute_force_contract=brute_force_contract,
        quantization=quantization,
        prediction_artifact=prediction_artifact,
        rate_artifact=rate_artifact,
        transition_artifact=transition_artifact,
        episode_readout=episode_readout,
        well_metrics=well_metrics,
        ledger=ledger,
        elapsed_seconds=elapsed_seconds,
    )
    gate_artifact = write_json(
        output_dir / f"{EXPERIMENT_NAME}_stage0_gate_report.json",
        gates,
    )
    rerun_contract = {
        "deterministic_anchor": False,
        "first_run_is_anchor": False,
        "required_before_anchor_reconsideration": [
            "identical_prediction_logical_sha256",
            "identical_transition_ledger_logical_sha256",
        ],
        "first_run_prediction_logical_sha256": prediction_artifact[
            "logical_sha256"
        ],
        "first_run_transition_ledger_logical_sha256": transition_artifact[
            "logical_sha256"
        ],
    }
    summary = {
        "experiment": EXPERIMENT_NAME,
        "route": "pf_beam",
        "status": (
            "stage0_pass_pending_separate_stage1_approval"
            if gates["stage0_all_gates_pass"]
            else "stage0_fail_closed"
        ),
        "scientific_variant": SCIENTIFIC_VARIANT,
        "execution_contract": execution_contract,
        "scientific_contract": scientific_contract,
        "runtime": runtime_versions(),
        "numba_threads": 1,
        "state_order": "well,row,u_position,u_rate,source_edge,destination_edge",
        "input_reports": {
            "fixed32": manifest_report,
            "saved_exp209": parent_report,
            "persistent_episodes": episode_input_report,
        },
        "numerical_contract": numerical_contract,
        "gates": gates,
        "reproducibility": rerun_contract,
        "artifacts": {
            "prediction": prediction_artifact,
            "rate_readout": rate_artifact,
            "transition_quantization": transition_artifact,
            "numerical_contract": numerical_contract_artifact,
            "well_metrics": well_artifact,
            "episode_metrics": episode_artifact,
            "gate_report": gate_artifact,
        },
        "elapsed_seconds": elapsed_seconds,
        "created_at_utc": pd.Timestamp.utcnow().isoformat(),
    }
    metrics_report = write_json(metrics_path(), summary)
    summary["artifacts"]["metrics"] = metrics_report
    print(json.dumps(to_jsonable(summary), indent=2, sort_keys=True))
    return summary

## 9. Configuration preview and guarded execution

Importing or opening the notebook never starts Stage 0. A Kaggle run requires
both `runtime.run_approved=true` and `execution.run_hmm=true`; the committed
implementation keeps both false until a separate user approval.

In [ ]:
if __name__ == "__main__":
    CONFIG = load_config()
    EXECUTION_PREVIEW = validate_execution_contract(
        CONFIG,
        require_run_authorization=False,
    )
    SCIENTIFIC_PREVIEW = validate_scientific_contract(CONFIG)
    print(
        json.dumps(
            {
                "experiment": EXPERIMENT_NAME,
                "route": get_nested(CONFIG, "experiment.route"),
                "status": get_nested(CONFIG, "experiment.status"),
                "execution": EXECUTION_PREVIEW,
                "scientific_variant": SCIENTIFIC_VARIANT,
                "parent": PARENT_EXPERIMENT,
                "continuous_coordinate_equivalence": "exact",
                "scientific_difference": (
                    "fixed_discrete_lattice_coordinate_only"
                ),
                "run_approved": get_nested(CONFIG, "runtime.run_approved"),
                "run_hmm": get_nested(CONFIG, "execution.run_hmm"),
            },
            indent=2,
            sort_keys=True,
        )
    )
    STAGE0_RESULT = run_stage0(CONFIG)